# Replications
Here we can determine the amount of replications by following the formula n => (s*z/error)^2 where we want our error to be 0.01

This step is performed AFTER determining the warmup period (determined number of warm up periods which I currently set to 90) and after determining the length of a replication.

In [10]:
import math
import os
import random

import openpyxl
from openpyxl import load_workbook

from simulation import Simulation

First fill in the configuration
- Warmup weeks
- run weeks (replication length = warmup weeks + run weeks)
- initial amount of replications (R)
- Excel file you want to write it in
- where to find the input files

Besides this we also implemented a nested lis twhere you give the amount of time slots, strategy, runs and we will loop over these configurations (will make the inputs automatically)

In [11]:
CONFIGURATIONS = [
    [12, 3, 1],
    [16, 1, 2],
    [14, 2, 4],
    [10, 2, 3],
    [20, 2, 1],
    [16, 3, 3],
    [14, 3, 2],
    [14, 1, 1],
    [10, 1, 2],
    [20, 1, 4],
]

WARMUP_WEEKS: int = 50     
RUN_WEEKS:    int = 483     
R:            int = 16     
EXCEL_PATH:   str = "Excel Files/replications_final_configurations_before_antithetic.xlsx"
INPUT_DIR:    str = "Inputs"

Helper functions

In [12]:
def safe_avg(values: list[float]) -> float:
    valid = [v for v in values if math.isfinite(v)]
    return sum(valid) / len(valid) if valid else 0.0


def sheet_name_for(config: list[int]) -> str:
    n_slots, strategy, rule = config
    return f"S{strategy}-{n_slots}slots-R{rule}"[:31]

main function

In [13]:
def run_all(configurations, warmup_weeks, run_weeks, R, excel_path, input_dir):
    #Determine if excel path exists to know if we need to edit the new path or ratherr create a completely new file
    os.makedirs(os.path.dirname(excel_path), exist_ok=True)
    if os.path.exists(excel_path):
        wb = load_workbook(excel_path)
    else:
        wb = openpyxl.Workbook()
        # remove the default empty sheet openpyxl creates
        wb.remove(wb.active)

    total_weeks = warmup_weeks + run_weeks

    for config in configurations:
        n_slots, strategy, rule = config
        filename = os.path.join(input_dir, f"input-S{strategy}-{n_slots}.txt")
        sname    = sheet_name_for(config)

        print(f"\n{'='*60}")
        print(f"Config: {n_slots} urgent slots | Strategy {strategy} | Rule {rule}")
        print(f"  File : {filename}")
        print(f"  Sheet: {sname}")
        print(f"{'='*60}")

        if sname in wb.sheetnames: #if the sheet exists use it and just change after first row
            ws = wb[sname]
            print("  Sheet exists — appending rows.")

            for row in ws.iter_rows(min_row=2, max_col=6): #we delete (from row 2) what was written in the first 6 columns only
                for cell in row:
                    cell.value = None
        else: #if the sheet doesnt exist just make a new excel file/sheet
            ws = wb.create_sheet(sname)
            ws.append([ # columns we will apply
                "Replication",
                "Avg ElAppWT (h)",
                "Avg UrScanWT (h)",
                "Avg ElScanWT (h)",
                "Avg OT (h)",
                "Weighted Obj",
            ])

        # apply our simulation
        sim = Simulation(filename, total_weeks, R, rule)
        sim.setWeekSchedule()

        #applying multiple replicaitons
        for r in range(R):
            sim.resetSystem()
            random.seed(r)
            sim.runOneSimulation()

            # because we don't want to take into account a the warmup, slice the list
            post_el_app  = sim.movingAvgElectiveAppWT[warmup_weeks : warmup_weeks + run_weeks]
            post_ur_scan = sim.movingAvgUrgentScanWT [warmup_weeks : warmup_weeks + run_weeks]
            post_el_scan = sim.movingAvgElectiveScanWT[warmup_weeks: warmup_weeks + run_weeks]
            post_ot      = sim.movingAvgOT            [warmup_weeks : warmup_weeks + run_weeks]

            # now we can safely calculate the statistics (calculating average of replication via helperfunction), each row is thus the average over the replication (without warmup)
            avg_el_app  = safe_avg(post_el_app)
            avg_ur_scan = safe_avg(post_ur_scan)
            avg_el_scan = safe_avg(post_el_scan)
            avg_ot      = safe_avg(post_ot)
            weighted    = avg_el_app * sim.weightEl + avg_ur_scan * sim.weightUr

            # Write to excel
            row_idx = 2 + r  #always start at the second row
            ws.cell(row=row_idx, column=1, value=r)
            ws.cell(row=row_idx, column=2, value=round(avg_el_app,  5))
            ws.cell(row=row_idx, column=3, value=round(avg_ur_scan, 5))
            ws.cell(row=row_idx, column=4, value=round(avg_el_scan, 5))
            ws.cell(row=row_idx, column=5, value=round(avg_ot,      5))
            ws.cell(row=row_idx, column=6, value=round(weighted,    6))

            print(f"  r={r:>3}  elApp={avg_el_app:.3f}  urScan={avg_ur_scan:.3f}"
                  f"  elScan={avg_el_scan:.3f}  OT={avg_ot:.3f}  OV={weighted:.4f}")

        wb.save(excel_path)
        print(f"{excel_path}")

    print(f"\ndone")


call the function

In [14]:
if __name__ == "__main__":
    run_all(
        configurations=CONFIGURATIONS,
        warmup_weeks=WARMUP_WEEKS,
        run_weeks=RUN_WEEKS,
        R=R,
        excel_path=EXCEL_PATH,
        input_dir=INPUT_DIR,
    )


Config: 12 urgent slots | Strategy 3 | Rule 1
  File : Inputs\input-S3-12.txt
  Sheet: S3-12slots-R1
  r=  0  elApp=15.094  urScan=2.652  elScan=0.074  OT=0.863  OV=0.3845
  r=  1  elApp=19.683  urScan=2.654  elScan=0.074  OT=0.845  OV=0.4120
  r=  2  elApp=15.301  urScan=2.659  elScan=0.074  OT=0.914  OV=0.3866
  r=  3  elApp=17.160  urScan=2.704  elScan=0.073  OT=0.896  OV=0.4026
  r=  4  elApp=15.932  urScan=2.729  elScan=0.074  OT=0.884  OV=0.3980
  r=  5  elApp=16.633  urScan=2.728  elScan=0.074  OT=0.935  OV=0.4021
  r=  6  elApp=16.635  urScan=2.697  elScan=0.074  OT=0.853  OV=0.3987
  r=  7  elApp=18.478  urScan=2.668  elScan=0.075  OT=0.878  OV=0.4064
  r=  8  elApp=17.914  urScan=2.703  elScan=0.075  OT=0.872  OV=0.4069
  r=  9  elApp=15.784  urScan=2.727  elScan=0.074  OT=0.933  OV=0.3970
  r= 10  elApp=17.571  urScan=2.643  elScan=0.075  OT=0.863  OV=0.3983
  r= 11  elApp=17.422  urScan=2.732  elScan=0.076  OT=0.874  OV=0.4073
  r= 12  elApp=15.237  urScan=2.646  elScan=0.